# Weight Pruning — Comet Sweep Analysis

Analyses for the 5 comet sweeps in `paper-weight-pruning-scaling-sweep`:

- **dense_fill_main**: block_sparse → dense fill experiment at n=16, h=256.
- **n_task_scaling**: block_sparse / dense / random_sparse / structure_search, each sweeping `task.n_tasks` × a size axis × LRs.

All sweeps log per-seed `(n_log_periods, n_seeds)` arrays as comet assets (`per_seed_losses.npy`, `per_seed_accs.npy`). Live metric series carry only the across-seed mean+std; the per-seed assets are loaded here to compute real 95% CI bands.

In [ ]:
# # Run once to fetch from Comet. Comment out after first run.
# from phd.research_utils.scripts.comet_download import run as _comet_download
# _comet_download(
#     project='paper-weight-pruning-scaling-sweep',
#     output_dir='data',
# )

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import io
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import t as t_dist

from comet_ml.api import API

from phd.research_utils.analysis.analysis_utils import (
    set_matplotlib_style, get_color_palette, save_fig_versions,
)
from phd.research_utils.analysis.plotting import (
    plot_param_sensitivity,
)

set_matplotlib_style('2-row')
%matplotlib inline

PROJECT = 'paper-weight-pruning-scaling-sweep'
SWEEP_IDS = {
    'dense_fill_main':    'dc6d3df68d25479999b72a32af29c275',
    'n_block_sparse':     'bf92bf7c1ced4535823e892b8049089d',
    'n_dense':            'b21316aca20d4aa7b60d05f03d59babc',
    'n_random_sparse':    'ac0163f45597446fbdea4a12edffd3ab',
    'n_structure_search': 'eb0849167ea44728a1173f9a794fd165',
}
SWEEP_NAMES = {
    'dense_fill_main':    '00_block_sparse_dense_fill_main_n16_h256',
    'n_block_sparse':     '00_block_sparse_n_task_scaling_lr_5seed',
    'n_dense':            '00_dense_n_task_scaling_lr_5seed',
    'n_random_sparse':    '00_random_sparse_n_task_scaling_lr_5seed',
    'n_structure_search': '00_structure_search_n_task_scaling_lr_5seed',
}

DATA_DIR    = Path('data')
ASSET_CACHE = DATA_DIR / 'per_seed_cache'
ASSET_CACHE.mkdir(parents=True, exist_ok=True)

def save_fig(name):
    save_fig_versions(name, dir='figures/pdf/', type='pdf')
    save_fig_versions(name, dir='figures/png/', type='png')

## Load data

`comet_download` dumps two CSVs: one with per-trial parameters, one with the metric history. The metric history uses Comet's `step` index, which the train script sets to the global training step at each log period.

In [ ]:
PARAMS_PATH  = DATA_DIR / f'{PROJECT}_params.csv'
METRICS_PATH = DATA_DIR / f'{PROJECT}_metrics.csv'

cfg_all = pd.read_csv(PARAMS_PATH, index_col=0)
run_all = pd.read_csv(METRICS_PATH, index_col=0)

# Numeric coercion for the columns we'll group on. (Comet stringifies these.)
_NUMERIC_COLS = [
    'task.n_tasks', 'model.initial_hidden_units', 'model.max_hidden_units',
    'model.dense_fill_step', 'structure_search.connection_budget',
    'structure_search.prune_count', 'structure_search.max_units_per_event',
    'optimizer.learning_rate', 'seed_offset',
]
for col in _NUMERIC_COLS:
    if col in cfg_all.columns:
        cfg_all[col] = pd.to_numeric(cfg_all[col], errors='coerce')

run_all = run_all.dropna(subset=['step'])

print(f'{len(cfg_all)} trials, {len(run_all)} metric rows')
print('\nsweep_name counts:')
print(cfg_all['sweep_name'].value_counts())

## Helpers

In [ ]:
def filter_sweep(key):
    """Slice cfg+run to one sweep by short key (e.g. 'n_block_sparse')."""
    target = SWEEP_NAMES[key]
    cfg = cfg_all[cfg_all['sweep_name'] == target].copy()
    run = run_all[run_all['run_id'].isin(cfg['run_id'])].copy()
    return cfg, run


def asymptotic_metric(cfg, run, metric_col, group_cols):
    """Mean of `metric_col` over the last 5% of training steps, aggregated per (group, LR).

    Each trial already represents the across-seed mean (Comet metric stream is
    mean+std). For per-seed CI you load assets via fetch_per_seed_arrays.
    """
    max_step = run.groupby('run_id')['step'].max()
    thr = (max_step * 0.95).rename('_thr')
    rdf = run.merge(thr, left_on='run_id', right_index=True)
    last5 = rdf[rdf['step'] >= rdf['_thr']]
    per_run = last5.groupby('run_id')[metric_col].mean().dropna().rename('_metric')
    merged = cfg.merge(per_run, left_on='run_id', right_index=True)
    return merged.groupby(group_cols + ['optimizer.learning_rate'])['_metric'].mean().reset_index()


def filter_to_best(cfg, run, group_cols, metric_col, direction='max'):
    """Keep only the rows whose LR is best within each group."""
    asym = asymptotic_metric(cfg, run, metric_col, group_cols)
    grouped = asym.groupby(group_cols)['_metric']
    idxs = grouped.idxmax() if direction == 'max' else grouped.idxmin()
    best = asym.loc[idxs]
    keep = set(map(tuple, best[group_cols + ['optimizer.learning_rate']].values))
    cfg_keys = list(map(tuple, cfg[group_cols + ['optimizer.learning_rate']].values))
    mask = pd.Series([k in keep for k in cfg_keys], index=cfg.index)
    return cfg[mask].copy(), run[run['run_id'].isin(cfg.loc[mask, 'run_id'])].copy(), best

In [ ]:
# Per-seed asset loader (cached + concurrent).
_api_singleton = None
def _api():
    global _api_singleton
    if _api_singleton is None:
        _api_singleton = API()
    return _api_singleton


def fetch_per_seed_arrays(run_ids, n_threads=8):
    """Download per_seed_losses + per_seed_accs for a set of run_ids; cached on disk.

    Returns: dict run_id -> {'losses': arr, 'accs': arr}, each (n_log_periods, n_seeds).
    """
    def fetch_one(run_id):
        cache_l = ASSET_CACHE / f'{run_id}_losses.npy'
        cache_a = ASSET_CACHE / f'{run_id}_accs.npy'
        if cache_l.exists() and cache_a.exists():
            return run_id, {'losses': np.load(cache_l), 'accs': np.load(cache_a)}
        try:
            e = _api().get_experiment_by_key(run_id)
            asset_list = e.get_asset_list()
            by_name = {a['fileName']: a for a in asset_list}
            for asset_name, cache_path in [
                ('per_seed_losses.npy', cache_l),
                ('per_seed_accs.npy',  cache_a),
            ]:
                if asset_name not in by_name:
                    return run_id, None
                data = e.get_asset(by_name[asset_name]['assetId'], return_type='binary')
                arr = np.load(io.BytesIO(data))
                np.save(cache_path, arr)
            return run_id, {'losses': np.load(cache_l), 'accs': np.load(cache_a)}
        except Exception as exc:
            print(f'failed for {run_id}: {exc}')
            return run_id, None

    out = {}
    with ThreadPoolExecutor(max_workers=n_threads) as ex:
        futures = {ex.submit(fetch_one, rid): rid for rid in run_ids}
        for f in tqdm(as_completed(futures), total=len(futures), desc='fetch per-seed'):
            rid, data = f.result()
            if data is not None:
                out[rid] = data
    return out


def stack_per_seed(asset_dict, run_ids):
    """Concat per-seed arrays across a list of run_ids along the seed axis.

    Used to merge dense_fill_main's two seed_offset halves into a single
    (n_periods, 30) view per (fill_step, LR) cell.
    Returns: (losses, accs) of shape (n_periods, n_combined_seeds).
    """
    present = [rid for rid in run_ids if rid in asset_dict]
    if not present:
        return None, None
    losses = np.concatenate([asset_dict[rid]['losses'] for rid in present], axis=1)
    accs   = np.concatenate([asset_dict[rid]['accs']   for rid in present], axis=1)
    return losses, accs


def plot_ci_curve(steps, arr, label=None, color=None, alpha=0.20, ax=None):
    """Plot mean + 95% t-CI band from a (n_periods, n_seeds) array."""
    ax = ax or plt.gca()
    mean = arr.mean(axis=1)
    n = arr.shape[1]
    sem = arr.std(axis=1, ddof=1) / np.sqrt(n)
    half_w = t_dist.ppf(0.975, df=n - 1) * sem
    ax.plot(steps, mean, label=label, color=color)
    ax.fill_between(steps, mean - half_w, mean + half_w, alpha=alpha, color=color, linewidth=0)

## 1. dense_fill_main

n_tasks=16, hidden_units=256, total_steps=500k. The grid is `(dense_fill_step ∈ {0, 250000, 600000}) × (LR ∈ {2^-11..2^-7}) × (seed_offset ∈ {0, 15})`. 600000 is the never-fill sentinel (block_sparse baseline), 0 = dense-from-start with cross-task weights initially 0.

In [ ]:
df_cfg, df_run = filter_sweep('dense_fill_main')
df_run['loss_per_task'] = df_run['loss'] / 16
print(f'{len(df_cfg)} trials')

# Sanity: should be 3 fill steps × 5 LRs × 2 seed_offsets = 30 trials.
print(df_cfg.groupby(['model.dense_fill_step', 'seed_offset']).size())

In [ ]:
plt.figure(figsize=(7.5, 4.5))
plot_param_sensitivity(
    run_df=df_run, config_df=df_cfg,
    x_col='optimizer.learning_rate',
    title='dense_fill_main — Asymptotic Accuracy vs LR',
    x_label='Learning rate', y_label='Asymptotic accuracy',
    metric_col='accuracy', metric_type='final_avg',
    hue_col='model.dense_fill_step', legend_title='dense_fill_step',
    pow_2_x_axis=True, nan_policy='omit',
)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Best LR per fill_step; pull the corresponding two seed_offset halves.
best_cfg, best_run, best = filter_to_best(
    df_cfg, df_run, group_cols=['model.dense_fill_step'],
    metric_col='accuracy', direction='max',
)
print('best LR per fill_step:')
print(best)

# Should be 6 trials (3 fill_step × 2 seed_offset).
asset_dict = fetch_per_seed_arrays(best_cfg['run_id'].tolist())

In [ ]:
fill_steps = sorted(best_cfg['model.dense_fill_step'].unique())
palette = sns.color_palette('flare', n_colors=len(fill_steps))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for fs, color in zip(fill_steps, palette):
    rids = best_cfg[best_cfg['model.dense_fill_step'] == fs]['run_id'].tolist()
    losses, accs = stack_per_seed(asset_dict, rids)
    if losses is None:
        continue
    steps = np.arange(losses.shape[0]) * 1000
    fs_label = 'null (no fill)' if fs >= 600000 else str(int(fs))
    plot_ci_curve(steps, accs,       label=f'fill={fs_label}', color=color, ax=axes[0])
    plot_ci_curve(steps, losses / 16, label=f'fill={fs_label}', color=color, ax=axes[1])
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (mean ± 95% CI)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss / n_tasks (mean ± 95% CI)')
axes[0].set_title('dense_fill_main — accuracy at best LR per fill_step')
axes[1].set_title('dense_fill_main — per-task loss at best LR per fill_step')
axes[0].legend(loc='best'); axes[1].legend(loc='best')
axes[0].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 2. n_task_scaling: 4-method comparison

All four methods sweep `task.n_tasks ∈ {2,4,8,16,32}` × a size axis × LRs. The size axis differs by method:

| Method | Size axis | Grid |
|---|---|---|
| Block-Sparse | `model.initial_hidden_units` | 4, 8, ..., 512 |
| Dense | `model.initial_hidden_units` | 4, 8, ..., 512 |
| Random-Sparse | `model.initial_hidden_units` | 48, 96, ..., 1536 (per-unit sparsity fixed at 03 winner) |
| Structure-Search | `structure_search.connection_budget` | 4764, ..., 152448 (max_hidden/prune/units proportional) |

In [ ]:
METHODS = [
    ('n_block_sparse',     'Block-Sparse',     'model.initial_hidden_units'),
    ('n_dense',            'Dense',            'model.initial_hidden_units'),
    ('n_random_sparse',    'Random-Sparse',    'model.initial_hidden_units'),
    ('n_structure_search', 'Structure-Search', 'structure_search.connection_budget'),
]
METHOD_PALETTE = get_color_palette(classes=[m[1] for m in METHODS])

for key, label, size_col in METHODS:
    c, r = filter_sweep(key)
    print(f'{label:18s}  trials={len(c):4d}  sizes={sorted(c[size_col].dropna().unique().tolist())}')

### 2a. Per-method: asymptotic accuracy vs size, hue = n_tasks (best LR per cell)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, (key, label, size_col) in zip(axes.flatten(), METHODS):
    cfg, run = filter_sweep(key)
    if len(cfg) == 0:
        ax.set_title(f'{label} — no data'); continue
    asym = asymptotic_metric(cfg, run, 'accuracy', group_cols=['task.n_tasks', size_col])
    best = asym.loc[asym.groupby(['task.n_tasks', size_col])['_metric'].idxmax()].copy()
    best = best.sort_values(['task.n_tasks', size_col])
    n_palette = sns.color_palette('viridis', n_colors=best['task.n_tasks'].nunique())
    for color, (n_t, sub) in zip(n_palette, best.groupby('task.n_tasks')):
        ax.plot(sub[size_col], sub['_metric'], '-o',
                color=color, label=f'n_tasks={int(n_t)}')
    ax.set_xscale('log', base=2)
    ax.set_xlabel(size_col)
    ax.set_ylabel('Asymptotic accuracy')
    ax.set_title(f'{label}: asymp. acc vs size (best LR per cell)')
    ax.set_ylim(0, 1)
    ax.legend(fontsize='x-small', loc='best')
plt.tight_layout()
plt.show()

### 2b. Cross-method: best-tuned asymptotic accuracy vs n_tasks

For each method, picks the best `(size, LR)` combination at each `n_tasks` value and plots that single point. Direct head-to-head.

In [ ]:
records = []
for key, label, size_col in METHODS:
    cfg, run = filter_sweep(key)
    if len(cfg) == 0:
        continue
    asym = asymptotic_metric(cfg, run, 'accuracy', group_cols=['task.n_tasks', size_col])
    best = asym.loc[asym.groupby(['task.n_tasks'])['_metric'].idxmax()]
    for _, r in best.iterrows():
        records.append({
            'Method': label, 'n_tasks': int(r['task.n_tasks']),
            'size': r[size_col], 'lr': r['optimizer.learning_rate'], 'asym_acc': r['_metric'],
        })
df_best = pd.DataFrame(records)

plt.figure(figsize=(8, 5))
for label in [m[1] for m in METHODS]:
    sub = df_best[df_best['Method'] == label].sort_values('n_tasks')
    plt.plot(sub['n_tasks'], sub['asym_acc'], '-o',
             color=METHOD_PALETTE[label], label=label)
plt.xscale('log', base=2)
plt.xticks([2, 4, 8, 16, 32], [2, 4, 8, 16, 32])
plt.xlabel('Number of tasks')
plt.ylabel('Asymptotic accuracy (best size × best LR)')
plt.title('n_task_scaling — best-tuned asymptotic accuracy per method')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

df_best

### 2c. Cross-method learning curves with 95% CI (n_tasks = 4, best size × best LR per method)

Per-seed `.npy` assets feed the CI bands. 5 atomic seeds per trial.

In [ ]:
TARGET_N_TASKS = 4
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for key, label, size_col in METHODS:
    cfg, run = filter_sweep(key)
    cfg = cfg[cfg['task.n_tasks'] == TARGET_N_TASKS].copy()
    run = run[run['run_id'].isin(cfg['run_id'])].copy()
    if len(cfg) == 0:
        continue
    asym = asymptotic_metric(cfg, run, 'accuracy', group_cols=[size_col])
    row = asym.loc[asym['_metric'].idxmax()]
    best_size, best_lr = row[size_col], row['optimizer.learning_rate']
    rids = cfg[(cfg[size_col] == best_size)
               & (cfg['optimizer.learning_rate'] == best_lr)]['run_id'].tolist()
    assets = fetch_per_seed_arrays(rids)
    losses, accs = stack_per_seed(assets, rids)
    if losses is None:
        continue
    steps = np.arange(losses.shape[0]) * 1000
    lr_log2 = int(np.log2(best_lr))
    legend = f'{label}  size={int(best_size)} lr=2^{lr_log2}'
    plot_ci_curve(steps, accs, label=legend, color=METHOD_PALETTE[label], ax=axes[0])
    plot_ci_curve(steps, losses / TARGET_N_TASKS, label=legend, color=METHOD_PALETTE[label], ax=axes[1])
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Accuracy (mean ± 95% CI)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss / n_tasks (mean ± 95% CI)')
axes[0].set_title(f'n_task_scaling @ n_tasks={TARGET_N_TASKS} — best (size, LR) per method')
axes[1].set_title(f'n_task_scaling @ n_tasks={TARGET_N_TASKS} — best (size, LR) per method')
axes[0].legend(fontsize='x-small', loc='best')
axes[1].legend(fontsize='x-small', loc='best')
axes[0].set_ylim(0, 1)
plt.tight_layout()
plt.show()

### 2d. LR-sensitivity per method (n_tasks = 4)

Verifies the LR grid spans the optimum at one representative n_tasks. Each panel shows asymptotic accuracy vs LR with hue = size axis.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (key, label, size_col) in zip(axes.flatten(), METHODS):
    cfg, run = filter_sweep(key)
    cfg = cfg[cfg['task.n_tasks'] == TARGET_N_TASKS].copy()
    run = run[run['run_id'].isin(cfg['run_id'])].copy()
    if len(cfg) == 0:
        ax.set_title(f'{label} — no data'); continue
    plt.sca(ax)
    plot_param_sensitivity(
        run_df=run, config_df=cfg,
        x_col='optimizer.learning_rate',
        title=f'{label} @ n_tasks={TARGET_N_TASKS}',
        x_label='Learning rate', y_label='Asymptotic accuracy',
        metric_col='accuracy', metric_type='final_avg',
        hue_col=size_col, legend_title=size_col,
        pow_2_x_axis=True, nan_policy='omit',
    )
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()